# sEMG Prosthetic Gesture Classification
## Notebook 08b: Deep-Learning Baseline (Google Colab Edition)

This notebook trains a real 1D convolutional neural network directly on raw,
filtered sEMG windows (12 channels x 400 samples, the same 200 ms / 50 ms
windows used throughout this project) as a deep-learning comparison point
for the classical CatBoost/XGBoost/LightGBM results in the manuscript.

**Why this notebook exists:** the manuscript's Introduction and Discussion
repeatedly contrast classical multi-domain feature pipelines against
"high-90s" deep-learning results quoted from the literature, without ever
running a deep network under the *same* subject-disjoint protocol on the
*same* data. This notebook closes that gap with a real, trained,
evaluated baseline rather than a literature-quoted number.

**Data:** the real segmented raw-window dataset (692,276 windows total,
exactly matching the classical feature pipeline's window count) was
generated locally via the project's existing `src/segmentation.py`
`SegmentationPipeline` over the preprocessed (filtered + normalized)
NinaPro DB2 recordings, then repackaged as float16 + compressed `.npz`
per subject for a feasible Colab upload. Global gesture labels (0-49)
come directly from the dataset's `stimulus` field and require no
remapping -- verified identical to the classical pipeline's `gesture_id`
column.

**Split:** the exact same subject-disjoint split used throughout this
project (`outputs/reports/split_metadata_top50.json`): 28 train / 6
validation / 6 held-out test subjects.

In [ ]:
# ==============================================================
# GOOGLE COLAB SETUP & ENVIRONMENT INITIALIZATION
# ==============================================================
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/semg-prosthetic-gesture-classification'
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"Changed working directory to Google Drive: {PROJECT_PATH}")
    else:
        raise FileNotFoundError(
            f"{PROJECT_PATH} not found. Upload/sync the project folder to this path first."
        )
    !pip install -q torch torchmetrics scikit-learn
else:
    PROJECT_PATH = str(Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd()))

print(f"IN_COLAB={IN_COLAB} | PROJECT_PATH={PROJECT_PATH}")


### Data required on Drive for this notebook

Upload `data/processed/segmented_raw_compact/subject_*.npz` (40 files, one
per subject; float16 + compressed, ~3-4 GB total) to the same relative
path under `MyDrive/semg-prosthetic-gesture-classification/`, in addition
to `outputs/reports/split_metadata_top50.json` (already present from
earlier notebooks).

In [ ]:
import json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = Path(PROJECT_PATH)
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "segmented_raw_compact"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
MODELS_DIR = PROJECT_ROOT / "models" / "optimized"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected -- training ~485k windows on CPU will be slow. "
          "Select Runtime -> Change runtime type -> T4 GPU.")

split_meta = json.load(open(REPORTS_DIR / "split_metadata_top50.json"))
train_subjects = split_meta["train_subjects"]
val_subjects = split_meta["val_subjects"]
test_subjects = split_meta["test_subjects"]
print(f"Train subjects ({len(train_subjects)}): {train_subjects}")
print(f"Val subjects ({len(val_subjects)}): {val_subjects}")
print(f"Test subjects ({len(test_subjects)}): {test_subjects}")


In [ ]:
class SEMGWindowDataset(Dataset):
    """Loads real raw sEMG windows for a list of subjects into memory.
    IMPORTANT: windows are kept as float16 in RAM (not upcast to float32
    for the whole resident array) -- with all three splits (28/6/6
    subjects) loaded at once, float32 residency is ~13.4 GB, which
    exceeds a standard Colab instance's system RAM and causes a silent
    OOM kernel restart (no Python traceback, just "kernel restarted" in
    the server log). float16 residency is ~6.7 GB total, which fits
    safely. Each individual sample is cast to float32 only when fetched,
    which is cheap (400x12 elements)."""
    def __init__(self, subject_ids, data_dir):
        windows_list, labels_list = [], []
        for sid in subject_ids:
            d = np.load(data_dir / f"subject_{sid}.npz")
            windows_list.append(d["windows"])  # keep as float16
            labels_list.append(d["labels"])
        self.windows = np.concatenate(windows_list, axis=0)  # stays float16
        self.labels = np.concatenate(labels_list, axis=0).astype(np.int64)
        del windows_list, labels_list
        print(f"  loaded {len(subject_ids)} subjects -> {self.windows.shape[0]} windows "
              f"({self.windows.nbytes / 1e9:.2f} GB resident, float16)")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.windows[idx].astype(np.float32).transpose(1, 0)  # (400, 12) -> (12, 400)
        return torch.from_numpy(x), int(self.labels[idx])


print("Loading train split...")
train_ds = SEMGWindowDataset(train_subjects, DATA_DIR)
print("Loading val split...")
val_ds = SEMGWindowDataset(val_subjects, DATA_DIR)
print("Loading test split...")
test_ds = SEMGWindowDataset(test_subjects, DATA_DIR)

# num_workers=0: with fork-based multiprocessing, worker subprocesses can
# still balloon memory further; since __getitem__ is pure in-memory numpy
# indexing (no disk I/O), single-process loading is fast enough and safer.
BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)


### Model: compact 1D CNN

Four convolutional blocks (Conv1d + BatchNorm + ReLU + MaxPool), global
average pooling, dropout, and a linear classifier over 50 classes --
a standard, unremarkable architecture for raw-sEMG classification,
deliberately not tuned or novel, so the comparison against the classical
pipeline reflects representation learning versus hand-crafted features,
not architecture search.

In [ ]:
class SEMG1DCNN(nn.Module):
    def __init__(self, n_channels=12, n_classes=50):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SEMG1DCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Total parameters: {n_params:,}")


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
criterion = nn.CrossEntropyLoss()

MAX_EPOCHS = 25
PATIENCE = 5
best_val_f1 = -1.0
epochs_no_improve = 0
best_state = None
history = []

for epoch in range(MAX_EPOCHS):
    t0 = time.perf_counter()
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            out = model(xb)
            val_preds.append(out.argmax(1).cpu().numpy())
            val_true.append(yb.numpy())
    val_preds = np.concatenate(val_preds)
    val_true = np.concatenate(val_true)
    val_acc = accuracy_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds, average="macro", zero_division=0)
    scheduler.step(val_f1)

    elapsed = time.perf_counter() - t0
    print(f"Epoch {epoch+1}/{MAX_EPOCHS}: train_loss={train_loss:.4f} val_acc={val_acc:.4f} val_macroF1={val_f1:.4f} ({elapsed:.1f}s)")
    history.append({"epoch": epoch+1, "train_loss": train_loss, "val_acc": val_acc, "val_macro_f1": val_f1})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no val macro-F1 improvement for {PATIENCE} epochs).")
            break

model.load_state_dict(best_state)
print(f"\nBest validation macro F1: {best_val_f1:.4f}")

import pandas as pd
pd.DataFrame(history).to_csv(TABLES_DIR / "dl_baseline_training_history.csv", index=False)


### Real held-out test evaluation

Same metrics as Table 3 (Accuracy, Macro F1, MCC, macro one-vs-rest
ROC-AUC, ECE, model size, inference latency) for direct comparability.

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef, roc_auc_score,
    precision_score, recall_score
)
import torch.nn.functional as F

model.eval()
test_preds, test_true, test_probs = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)
        probs = F.softmax(out, dim=1)
        test_preds.append(out.argmax(1).cpu().numpy())
        test_probs.append(probs.cpu().numpy())
        test_true.append(yb.numpy())
test_inference_time = time.perf_counter() - t0
test_preds = np.concatenate(test_preds)
test_true = np.concatenate(test_true)
test_probs = np.concatenate(test_probs)

test_acc = accuracy_score(test_true, test_preds)
test_macro_f1 = f1_score(test_true, test_preds, average="macro", zero_division=0)
test_macro_precision = precision_score(test_true, test_preds, average="macro", zero_division=0)
test_macro_recall = recall_score(test_true, test_preds, average="macro", zero_division=0)
test_weighted_f1 = f1_score(test_true, test_preds, average="weighted", zero_division=0)
test_mcc = matthews_corrcoef(test_true, test_preds)
test_roc_auc = roc_auc_score(test_true, test_probs, multi_class="ovr", average="macro")

# Expected Calibration Error (10 bins), matching the classical pipeline's definition
confidences = test_probs.max(axis=1)
correct = (test_preds == test_true).astype(float)
bins = np.linspace(0, 1, 11)
ece = 0.0
for i in range(10):
    mask = (confidences > bins[i]) & (confidences <= bins[i+1])
    if mask.sum() > 0:
        ece += (mask.sum() / len(confidences)) * abs(correct[mask].mean() - confidences[mask].mean())

# single-sample latency (GPU/CPU as available, matches deployment-style measurement)
model_cpu = SEMG1DCNN().to("cpu")
model_cpu.load_state_dict(model.state_dict())
model_cpu.eval()
sample = torch.from_numpy(test_ds.windows[0].transpose(1,0)).unsqueeze(0).float()
n_reps = 300
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(n_reps):
        _ = model_cpu(sample)
single_sample_latency_ms = (time.perf_counter() - t0) / n_reps * 1000

n_params = sum(p.numel() for p in model.parameters())
model_size_mb = n_params * 4 / 1e6  # float32 parameters

print(f"Test Accuracy:        {test_acc:.4f}")
print(f"Test Macro F1:        {test_macro_f1:.4f}")
print(f"Test MCC:             {test_mcc:.4f}")
print(f"Test ROC-AUC (macro): {test_roc_auc:.4f}")
print(f"Test ECE:             {ece:.4f}")
print(f"Model size (MB):      {model_size_mb:.2f}")
print(f"Single-sample latency (CPU, ms): {single_sample_latency_ms:.3f}")
print(f"Batch inference time (test set, s): {test_inference_time:.2f}")

dl_results = {
    "Model": "1D-CNN",
    "Accuracy": test_acc,
    "Balanced Accuracy": test_macro_recall,
    "Macro Precision": test_macro_precision,
    "Macro Recall": test_macro_recall,
    "Macro F1": test_macro_f1,
    "Weighted F1": test_weighted_f1,
    "MCC": test_mcc,
    "ROC-AUC": test_roc_auc,
    "ECE": ece,
    "Model Size (MB)": model_size_mb,
    "Single-Sample Latency (ms)": single_sample_latency_ms,
    "N Parameters": n_params,
    "N Train Windows": len(train_ds),
    "N Val Windows": len(val_ds),
    "N Test Windows": len(test_ds),
    "Epochs Trained": len(history),
}
with open(TABLES_DIR / "dl_baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(dl_results, f, indent=2)
pd.DataFrame([dl_results]).to_csv(TABLES_DIR / "dl_baseline_results.csv", index=False)

torch.save(model.state_dict(), MODELS_DIR / "CNN1D_baseline.pt")
print("\nSaved dl_baseline_results.{json,csv} and CNN1D_baseline.pt")


### After completion

Download `outputs/tables/dl_baseline_results.json`, `dl_baseline_results.csv`,
`dl_baseline_training_history.csv`, and `models/optimized/CNN1D_baseline.pt`
back to the local project (same relative paths). These feed directly into
the manuscript's Results/Discussion update adding this deep-learning
baseline to Table 3 and the surrounding narrative.